### I’ll use a small custom adapter so you can see exactly what is trained:

$ 
                        h_{\text{out}}=h+W_{\text{up}}\operatorname{GELU}(W_{\text{down}}h+b_{\text{down}})+b_{\text{up}}
$
### All original model parameters stay frozen. Only the inserted adapters learn.

In [1]:
import hashlib
import math
import random

import pandas as pd
import torch
import torch.nn as nn

from IPython.display import display
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

# Choose "cpu", "cuda", or "auto".
DEVICE_CHOICE = "cpu"

if DEVICE_CHOICE == "auto":
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
elif DEVICE_CHOICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable. Choose "cpu".')
    device = torch.device("cuda")
elif DEVICE_CHOICE == "cpu":
    device = torch.device("cpu")
else:
    raise ValueError('Choose "cpu", "cuda", or "auto".')

random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

MODEL_NAME = "./workshop_models/distilgpt2"

BOTTLENECK_DIM = 16
MAX_LENGTH = 96
BATCH_SIZE = 1
TRAINING_STEPS = 30
LEARNING_RATE = 5e-4

print("Device:", device)
print("Bottleneck dimension:", BOTTLENECK_DIM)

Device: cpu
Bottleneck dimension: 16


In [23]:
# Domain dataset

In [24]:
train_texts = [
    "Predictive maintenance uses equipment condition data to plan "
    "maintenance before a failure occurs.",

    "Preventive maintenance follows a planned schedule based on "
    "time or equipment usage.",

    "Vibration monitoring helps engineers identify changes in "
    "the condition of rotating machinery.",

    "Bearing faults can produce characteristic vibration patterns. "
    "Engineers examine frequency information and operating conditions.",

    "An increase in bearing temperature may indicate lubrication "
    "problems, increased load, or other developing issues.",

    "Root mean square vibration summarizes the magnitude of a "
    "vibration signal over a measurement interval.",

    "A condition monitoring baseline describes normal equipment "
    "behavior under known operating conditions.",

    "Sensor trends should be compared with operating speed and load "
    "because these factors can influence measurements.",

    "A single abnormal reading does not confirm a machine fault. "
    "Additional measurements help support a diagnosis.",

    "Lubrication reduces friction and wear between moving surfaces. "
    "Incorrect lubrication can contribute to bearing damage.",

    "Shaft misalignment can increase vibration and mechanical stress "
    "in rotating equipment.",

    "Maintenance decisions should combine sensor measurements, "
    "inspection findings, and equipment history.",
]

eval_texts = [
    "Condition-based maintenance uses observations of machine health "
    "to decide when maintenance is needed.",

    "A technician investigating elevated vibration should consider "
    "changes in load, speed, alignment, and bearing condition.",

    "Rising bearing temperature is a symptom that requires further "
    "investigation rather than an immediate fault diagnosis.",

    "Comparing measurements with a normal operating baseline helps "
    "identify changes in equipment condition.",
]

assert set(train_texts).isdisjoint(eval_texts)

print("Training examples:", len(train_texts))
print("Evaluation examples:", len(eval_texts))

Training examples: 12
Evaluation examples: 4


In [25]:
## Load the model and prepare data

In [26]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# tokenizer.pad_token = tokenizer.eos_token

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     attn_implementation="eager",
# ).to(device)

# model.config.pad_token_id = tokenizer.pad_token_id
# model.config.use_cache = False


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    attn_implementation="eager",
).to(device)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False



def collate_texts(texts):
    batch = tokenizer(
        [text + tokenizer.eos_token for text in texts],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels

    return batch


train_loader = DataLoader(
    train_texts,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_texts,
)

eval_loader = DataLoader(
    eval_texts,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_texts,
)

print("Model loaded.")

Model loaded.


In [27]:
# Evaluation functions and original baseline

In [28]:
@torch.no_grad()
def evaluate_loss():
    model.eval()

    loss_sum = 0.0
    token_count = 0

    for batch in eval_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        output = model(**batch)

        valid_tokens = (
            batch["labels"][:, 1:] != -100
        ).sum().item()

        loss_sum += output.loss.item() * valid_tokens
        token_count += valid_tokens

    return loss_sum / token_count


@torch.no_grad()
def complete_text(prompt):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)

    generated = model.generate(
        **inputs,
        max_new_tokens=45,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]

    return tokenizer.decode(
        generated[0, input_length:],
        skip_special_tokens=True,
    ).strip()


test_prompts = [
    "Predictive maintenance is",
    "When bearing temperature increases,",
    "Vibration monitoring helps engineers",
]

baseline_loss = evaluate_loss()

baseline_answers = {
    prompt: complete_text(prompt)
    for prompt in test_prompts
}

print(f"Baseline loss: {baseline_loss:.4f}")

for prompt, answer in baseline_answers.items():
    print("\nPROMPT:", prompt)
    print("CONTINUATION:", answer)

Baseline loss: 5.4397

PROMPT: Predictive maintenance is
CONTINUATION: a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is

PROMPT: When bearing temperature increases,
CONTINUATION: the temperature increases, and the temperature decreases.

PROMPT: Vibration monitoring helps engineers
CONTINUATION: to understand the impact of the stress on the brain.


In [29]:
# Build the adapter
# Compression → activation → expansion → residual addition.

In [30]:
# Define a small neural network that learns a correction to hidden states.

class BottleneckAdapter(nn.Module):
    def __init__(self, hidden_dim, bottleneck_dim):
        # Initialize the PyTorch module.
        super().__init__()
         # Compress each token's hidden vector: 768 → 16 in this notebook.
        self.down = nn.Linear(hidden_dim, bottleneck_dim)
         # Apply a nonlinear transformation to the compressed representation.
        self.activation = nn.GELU()
        # Expand back to the original hidden dimension: 16 → 768.
        self.up = nn.Linear(bottleneck_dim, hidden_dim)

        # Initialize the expansion weights and bias to zero.
        # Initially, the adapter produces a zero correction.
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, hidden_states):
        # Compress the hidden states.
        correction = self.down(hidden_states)
        # Apply the nonlinear activation.
        correction = self.activation(correction)
        # Expand the correction to match the original hidden-state shape.
        correction = self.up(correction)
        # Add the learned correction to the original hidden states.
        # This residual connection preserves the original information.
        return hidden_states + correction


example_adapter = BottleneckAdapter(
    hidden_dim=model.config.n_embd,
    bottleneck_dim=BOTTLENECK_DIM,
)

example_input = torch.randn(1, 5, model.config.n_embd)

print("Input:", tuple(example_input.shape))
print("Compressed:", tuple(example_adapter.down(example_input).shape))
print("Output:", tuple(example_adapter(example_input).shape))

assert torch.equal(
    example_input,
    example_adapter(example_input),
)

print("Initialized adapter preserves the input.")

Input: (1, 5, 768)
Compressed: (1, 5, 16)
Output: (1, 5, 768)
Initialized adapter preserves the input.


In [31]:
# Insert adapters and freeze the original model
## placing one adapter after each transformer block.

In [32]:
# Prevent accidentally adding adapters twice.
assert all(
    not hasattr(block, "domain_adapter")
    for block in model.transformer.h
), "Adapters already exist. Rerun Cells 4–7 to start fresh."


# Freeze every existing model parameter BEFORE adding adapters.
# The original attention, feed-forward, and embedding weights cannot update.
for parameter in model.parameters():
    parameter.requires_grad = False

# This hook runs automatically after a transformer block finishes.
def adapter_hook(module, inputs, output):
    # A block may return hidden states together with other outputs.
    if isinstance(output, tuple):
        # Apply the adapter to the hidden states, stored at position 0.
        hidden_states = module.domain_adapter(output[0])
        # Replace the hidden states while preserving all other outputs.
        return (hidden_states,) + output[1:]
     # If the output is a single tensor, apply the adapter directly.
    return module.domain_adapter(output)

# Keep references to the hooks so they can be removed later if needed.
hook_handles = []

# DistilGPT2 has six transformer blocks.
for block in model.transformer.h:
    # Register a new adapter as a child module of this block.
    # Newly created adapter parameters are trainable by default.
    block.add_module(
        "domain_adapter",
        BottleneckAdapter(
            hidden_dim=model.config.n_embd, # Use the base model's hidden dimension: 768.
            bottleneck_dim=BOTTLENECK_DIM,# Use the selected bottleneck dimension: 16.
        ).to(device),
    )
    # Arrange for the adapter to process this block's output automatically.
    hook_handles.append(
        block.register_forward_hook(adapter_hook)
    )

# Collect parameters that are allowed to update.
# The original weights are frozen, so only adapter parameters qualify.
trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

# Verify that every trainable parameter belongs to an adapter.
assert all(
    "domain_adapter" in name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
)

total_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in trainable_parameters)

print("Adapters:", len(model.transformer.h))
print(f"Trainable parameters: {trainable_count:,}")
print(f"Trainable percentage: {100 * trainable_count / total_count:.4f}%")

display(pd.DataFrame([
    {
        "parameter": name,
        "shape": str(tuple(parameter.shape)),
    }
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]))

Adapters: 6
Trainable parameters: 152,160
Trainable percentage: 0.1854%


,parameter,shape
0,transformer.h.0.domain_adapter.down.weight,"(16, 768)"
1,transformer.h.0.domain_adapter.down.bias,"(16,)"
2,transformer.h.0.domain_adapter.up.weight,"(768, 16)"
3,transformer.h.0.domain_adapter.up.bias,"(768,)"
4,transformer.h.1.domain_adapter.down.weight,"(16, 768)"
5,transformer.h.1.domain_adapter.down.bias,"(16,)"
6,transformer.h.1.domain_adapter.up.weight,"(768, 16)"
7,transformer.h.1.domain_adapter.up.bias,"(768,)"
8,transformer.h.2.domain_adapter.down.weight,"(16, 768)"
9,transformer.h.2.domain_adapter.down.bias,"(16,)"


In [33]:
# Check initialization and fingerprint parameters

In [34]:
initialized_loss = evaluate_loss()

print(f"Original loss:         {baseline_loss:.6f}")
print(f"Initialized adapters:  {initialized_loss:.6f}")

assert math.isclose(
    baseline_loss,
    initialized_loss,
    rel_tol=1e-5,
    abs_tol=1e-5,
)


def frozen_fingerprints():
    return {
        name: hashlib.sha256(
            parameter.detach().cpu().contiguous().numpy().tobytes()
        ).hexdigest()
        for name, parameter in model.named_parameters()
        if not parameter.requires_grad
    }


frozen_before = frozen_fingerprints()

adapter_before = {
    name: parameter.detach().cpu().clone()
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
}

print("Verified: initialized adapters preserve baseline performance.")

Original loss:         5.439722
Initialized adapters:  5.439722
Verified: initialized adapters preserve baseline performance.


In [35]:
# Train the adapters

In [36]:
# Give only adapter parameters to the optimizer.
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.0,
)

training_history = []
iterator = iter(train_loader)

model.train() # Enable training mode; this does not unfreeze the original weights.


for step in range(1, TRAINING_STEPS + 1):
    try:
        batch = next(iterator)
    except StopIteration:
        iterator = iter(train_loader)
        batch = next(iterator)

    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }
    # Inside the training loop:
    optimizer.zero_grad(set_to_none=True)# Clear gradients from the previous step.
    # Run the model on the training batch.
    # Each transformer block's hook automatically applies its adapter.
    # Calculate next-token prediction loss using the supplied labels.
    loss = model(**batch).loss
    # Calculate gradients for the trainable adapter parameters.
    # Gradients pass through the model, but its original weights remain frozen.    
    loss.backward()
    # Limit adapter-gradient magnitude to help stabilize training.
    torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        max_norm=1.0,
    )

    optimizer.step() # Actually update the adapter weights and biases.


    training_history.append({
        "step": step,
        "training_loss": loss.item(),
    })

    if step == 1 or step % 5 == 0 or step == TRAINING_STEPS:
        print(
            f"Step {step}/{TRAINING_STEPS} "
            f"| Loss: {loss.item():.4f}"
        )

Step 1/30 | Loss: 5.3644
Step 5/30 | Loss: 5.0275
Step 10/30 | Loss: 4.6635
Step 15/30 | Loss: 4.0839
Step 20/30 | Loss: 4.4028
Step 25/30 | Loss: 3.1878
Step 30/30 | Loss: 3.9078


In [37]:
# Compare before and after

In [38]:
adapted_loss = evaluate_loss()

display(pd.DataFrame([
    {
        "stage": "Before adapter training",
        "evaluation_loss": baseline_loss,
        "perplexity": math.exp(baseline_loss),
    },
    {
        "stage": "After adapter training",
        "evaluation_loss": adapted_loss,
        "perplexity": math.exp(adapted_loss),
    },
]))

print(
    "Held-out loss reduction:",
    f"{baseline_loss - adapted_loss:.4f}",
)

for prompt in test_prompts:
    print("\nPROMPT:", prompt)
    print("\nBEFORE:", baseline_answers[prompt])
    print("\nAFTER:", complete_text(prompt))
    print("=" * 70)

,stage,evaluation_loss,perplexity
0,Before adapter training,5.439722,230.378021
1,After adapter training,4.096039,60.101760


Held-out loss reduction: 1.3437

PROMPT: Predictive maintenance is

BEFORE: a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is

AFTER: a condition that can help determine the condition of a machine.

PROMPT: When bearing temperature increases,

BEFORE: the temperature increases, and the temperature decreases.

AFTER: a mechanical fault can cause mechanical fault faults.

PROMPT: Vibration monitoring helps engineers

BEFORE: to understand the impact of the stress on the brain.

AFTER: understand the condition of a system.


In [39]:
### ositive loss reduction means improvement on these evaluation sentences. The small dataset and short run may produce modest changes or no improvement.

In [40]:
# Verifying only adapters changed

In [41]:
# Recalculate fingerprints of the frozen original parameters.
frozen_after = frozen_fingerprints()

# Identify any frozen parameter tensors whose values changed.
changed_frozen = [
    name
    for name in frozen_before
    if frozen_before[name] != frozen_after[name]
]

change_rows = []

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        after = parameter.detach().cpu()
        difference = after - adapter_before[name]

        change_rows.append({
            "parameter": name,
            "changed": not torch.equal(after, adapter_before[name]),
            "mean_absolute_change": difference.abs().mean().item(),
        })

change_df = pd.DataFrame(change_rows)
display(change_df)

# Confirm that no original model parameter tensor changed.
assert len(changed_frozen) == 0
# Confirm that at least one adapter parameter tensor changed.
# change_df was created by comparing adapters before and after training.
assert change_df["changed"].any()

print("Verified: adapters changed; original model parameters stayed identical.")

,parameter,changed,mean_absolute_change
0,transformer.h.0.domain_adapter.down.weight,True,0.003572
1,transformer.h.0.domain_adapter.down.bias,True,0.005919
2,transformer.h.0.domain_adapter.up.weight,True,0.003564
3,transformer.h.0.domain_adapter.up.bias,True,0.003628
4,transformer.h.1.domain_adapter.down.weight,True,0.004300
5,transformer.h.1.domain_adapter.down.bias,True,0.006952
6,transformer.h.1.domain_adapter.up.weight,True,0.003418
7,transformer.h.1.domain_adapter.up.bias,True,0.003794
8,transformer.h.2.domain_adapter.down.weight,True,0.004193
9,transformer.h.2.domain_adapter.down.bias,True,0.006279


Verified: adapters changed; original model parameters stayed identical.


In [42]:
# ## Particpant Exervise

1. Increase BOTTLENECK_DIM from 16 to 32.
   How does the trainable parameter count change?

2. Compare 30 and 60 training steps.

3. Explain why the adapter output has the same shape as its input.

4. What does the residual connection do?

5. Why is the up-projection initialized to zero?

6. Does better language-model loss guarantee factual accuracy?

For each fresh experiment:
Change settings in Cell 2, then rerun Cells 4–11.

## Key takeaway

Bottleneck adapters add small trainable transformations
inside a frozen model.

SyntaxError: invalid character '–' (U+2013) (79698367.py, line 17)